In [ ]:
# =========================
# 1. INSTALL LIBRARIES
# =========================
!pip install xgboost joblib pandas scikit-learn matplotlib -q

In [ ]:
# =============================
# 2. IMPORT LIBRARIES
# =============================
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from google.colab import files
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# =============================
# 3. UPLOAD CSV
# =============================
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())
df.head()


Saving training_data.csv to training_data.csv


,post_id,day_of_week,hour_of_day,is_weekend,content_length,has_image,has_video,reactions_count,comments_count,saves_count,engagement_score
0,69e8e5288aa53129cfe9f7aa,4,8,0,48,0,0,0,3,0,6
1,69e8e52a8aa53129cfe9f7b9,1,22,0,75,0,0,45,12,10,109
2,69e8e53a8aa53129cfe9f874,1,11,0,135,0,0,9,2,0,13
3,69e8e53d8aa53129cfe9f899,6,6,1,133,0,0,1,3,0,7
4,69e8e53f8aa53129cfe9f8a9,5,7,0,132,0,0,3,3,0,9


In [ ]:
# =============================
# 4. Features and Target
# =============================
FEATURES = [
    "dayOfWeek",
    "hour",
    "isWeekend",
    "captionLength",
    "captionWordCount",
    "hasCaption",
    "mediaType_image",
    "mediaType_video",
    "followerCountAtPostTime",
    "accountAgeDays",
    "userPostCountBefore",
    "userAvgEngagementBefore",
    "userAvgEngagementLast7Posts",
    "userAvgEngagementLast30Days",
    "userBestHourBefore",
    "userBestDayOfWeekBefore",
    "globalAvgEngagementByHour",
    "globalAvgEngagementByDayOfWeek",
    "globalAvgEngagementByDayHour"
]

TARGET = "engagementScore"
TIME_COLUMN = "createdAt"

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =============================
# 5. Check for missing data columns
# =============================
required_columns = FEATURES + [TARGET, TIME_COLUMN]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

print("All required columns exist.")

In [ ]:
# =============================
# 6. Data Cleaning
# =============================
df[TIME_COLUMN] = pd.to_datetime(df[TIME_COLUMN], errors="coerce")

df = df.dropna(subset=[TIME_COLUMN, TARGET])

df[FEATURES] = df[FEATURES].fillna(0)

for col in FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df = df.sort_values(TIME_COLUMN).reset_index(drop=True)

print("Rows after cleaning:", len(df))
df.head()

In [ ]:
# =============================
# 7. Split train and test
# =============================
n = len(df)

train_end = int(n * 0.80)
val_end = int(n * 0.90)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_val = val_df[FEATURES]
y_val = val_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

print("Total:", n)
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("Train time:", train_df[TIME_COLUMN].min(), "->", train_df[TIME_COLUMN].max())
print("Val time:", val_df[TIME_COLUMN].min(), "->", val_df[TIME_COLUMN].max())
print("Test time:", test_df[TIME_COLUMN].min(), "->", test_df[TIME_COLUMN].max())

In [ ]:
# =============================
# 8. Train model
# =============================
model = XGBRegressor(
    n_estimators=1000,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

In [ ]:
# =============================
# 9. Evaluate Model
# =============================
def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"===== {name} =====")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R2  :", r2)

val_pred = model.predict(X_val)
test_pred = model.predict(X_test)

evaluate_model("Validation", y_val, val_pred)
evaluate_model("Test", y_test, test_pred)

In [ ]:
# =============================
# 10. Feature importance
# =============================
importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

importance_df

plt.figure(figsize=(10, 6))
plt.barh(importance_df["feature"], importance_df["importance"])
plt.gca().invert_yaxis()
plt.title("Feature Importance")
plt.xlabel("Importance")
plt.show()

In [ ]:
# =============================
# 11. Save Model
# =============================
joblib.dump(model, "best_time_to_post_xgboost.pkl")
joblib.dump(FEATURES, "best_time_features.pkl")

print("Saved:")
print("best_time_to_post_xgboost.pkl")
print("best_time_features.pkl")

In [ ]:
# =============================
# 12. Download Model
# =============================
files.download("best_time_to_post_xgboost.pkl")
files.download("best_time_features.pkl")

In [ ]:
# =============================
# 13. Test predict 1 sample
# =============================
sample = X_test.iloc[[0]]

predicted_score = model.predict(sample)[0]

print("Sample input:")
display(sample)

print("Predicted engagement score:", predicted_score)
print("Actual engagement score:", y_test.iloc[0])

In [ ]:
# =============================
# 14. Test top 3 best time sample
# =============================
base_row = X_test.iloc[0].copy()

candidate_rows = []

for day in range(7):
    for hour in range(24):
        row = base_row.copy()
        row["dayOfWeek"] = day
        row["hour"] = hour
        row["isWeekend"] = 1 if day in [5, 6] else 0
        candidate_rows.append(row)

candidate_df = pd.DataFrame(candidate_rows)

scores = model.predict(candidate_df[FEATURES])

result_df = candidate_df.copy()
result_df["predicted_engagement_score"] = scores

top_3 = result_df.sort_values(
    "predicted_engagement_score",
    ascending=False
).head(3)

top_3[["dayOfWeek", "hour", "predicted_engagement_score"]]